In [17]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import SpectralClustering
from sklearn.ensemble import RandomForestClassifier

In [2]:
nnary_df = pd.read_csv("./output/nnary_features_enhanced.csv").set_index("filename")
nnary_df

,uni_1_aspect,uni_2_aspect,uni_3_aspect,uni_4_aspect,uni_5_aspect,uni_6_aspect,uni_1_count,uni_1_total_area,uni_1_avg_x,uni_1_avg_y,...,tri_5_7_8_count,tri_5_7_8_area_ratio_ij,tri_5_7_8_area_ratio_jk,tri_5_7_8_avg_x,tri_5_7_8_avg_y,tri_6_7_8_count,tri_6_7_8_area_ratio_ij,tri_6_7_8_area_ratio_jk,tri_6_7_8_avg_x,tri_6_7_8_avg_y
filename,,,,,,,,,,,,,,,,,,,,,
40795.png,1.203013,1.512195,2.101449,0.365854,1.173913,2.000000,2.0,3396.0,-38.604240,53.561837,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60259.png,NaN,0.886364,0.935484,0.204545,1.045455,1.409091,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45409.png,1.054348,0.897959,1.174419,0.452993,1.937500,1.476190,2.0,4462.0,-2.314433,115.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22733.png,0.660377,0.625000,1.017544,2.857143,0.750000,0.285714,1.0,1855.0,43.500000,80.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25886.png,1.162162,0.704918,0.563025,3.400000,1.421053,1.450000,1.0,1591.0,72.500000,46.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26753.png,0.897959,0.849057,0.598291,0.729798,1.181818,0.621622,1.0,2135.5,39.887224,53.269570,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42387.png,1.414634,1.511111,0.725352,0.487805,0.946429,1.322581,1.0,2378.0,-76.000000,37.500000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19466.png,1.488889,NaN,1.028571,2.918182,1.500000,0.852941,2.0,3132.0,49.873563,51.637931,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
nnary_df = nnary_df.fillna(0)

nnary_df = nnary_df.loc[:, ~(nnary_df == 0).all()]

In [4]:
nnary_sans78_df = nnary_df.loc[:, ~nnary_df.columns.str.contains(r'7|8')]

In [5]:
X = nnary_sans78_df[:40000].values
scaler = StandardScaler()
X = np.nan_to_num(X, nan=0.0)
X_scaled = scaler.fit_transform(X)



In [6]:
def compute_sparse_affinity_sklearn(X, sigma='auto', n_neighbors=100):
    """
    Compute sparse affinity matrix using sklearn's NearestNeighbors.
    More efficient than manual batch processing for moderate-sized datasets.
    """
    n_samples = X.shape[0]
    
    # Find k-nearest neighbors
    print("Finding nearest neighbors...")
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1, n_jobs=-1)
    nn.fit(X)
    distances, indices = nn.kneighbors(X)
    
    # Remove self-neighbors
    distances = distances[:, 1:]
    indices = indices[:, 1:]
    
    # Auto-compute sigma if needed
    if sigma == 'auto':
        # Use the median of k-th nearest neighbor distances
        k_distances = distances[:, -1]  # k-th nearest neighbor distance
        sigma = np.median(k_distances)
        print(f"Auto-computed sigma: {sigma:.4f}")
    
    # Compute affinities
    print("Computing affinities...")
    affinities = np.exp(-distances**2 / (2 * sigma**2))
    
    # Build sparse matrix
    print("Building sparse affinity matrix...")
    row_indices = np.repeat(np.arange(n_samples), n_neighbors)
    col_indices = indices.flatten()
    data = affinities.flatten()
    
    # Create sparse matrix
    affinity_matrix = csr_matrix(
        (data, (row_indices, col_indices)),
        shape=(n_samples, n_samples)
    )
    
    # Make symmetric
    affinity_matrix = affinity_matrix + affinity_matrix.T
    affinity_matrix = affinity_matrix.multiply(0.5)  # Average the weights
    
    return affinity_matrix

In [7]:
def compute_sparse_affinity_batch(X, sigma='auto', n_neighbors=100, batch_size=2000):
    n_samples = X.shape[0]
    
    # Use sklearn method if dataset fits in memory
    if n_samples <= 10000:
        return compute_sparse_affinity_sklearn(X)
    

    # Initialize data structures for sparse matrix
    row_indices = []
    col_indices = []
    data_values = []
    
    # Auto-compute sigma if needed
    if sigma == 'auto':
        sample_size = min(2000, n_samples)
        sample_idx = np.random.choice(n_samples, sample_size, replace=False)
        sample_nn = NearestNeighbors(n_neighbors=n_neighbors + 1, n_jobs=-1)
        sample_nn.fit(X[sample_idx])
        sample_distances, _ = sample_nn.kneighbors(X[sample_idx])
        k_distances = sample_distances[:, -1]
        sigma = np.median(k_distances)
        print(f"Auto-computed sigma: {sigma:.4f}")
    
    # Process in batches
    for i in range(0, n_samples, batch_size):
        end_i = min(i + batch_size, n_samples)
        batch_size_i = end_i - i
        
        print(f"Processing batch {i//batch_size + 1}/{(n_samples-1)//batch_size + 1}")
        
        # Find neighbors for this batch
        nn = NearestNeighbors(n_neighbors=min(n_neighbors + 1, n_samples), n_jobs=-1)
        nn.fit(X)
        distances, indices = nn.kneighbors(X[i:end_i])
        
        # Process each point in the batch
        for j in range(batch_size_i):
            global_idx = i + j
            
            # Remove self-neighbor
            mask = indices[j] != global_idx
            neighbor_indices = indices[j][mask][:n_neighbors]
            neighbor_distances = distances[j][mask][:n_neighbors]
            
            # Compute affinities
            affinities = np.exp(-neighbor_distances**2 / (2 * sigma**2))
            
            # Store for sparse matrix
            row_indices.extend([global_idx] * len(neighbor_indices))
            col_indices.extend(neighbor_indices)
            data_values.extend(affinities)
    
    # Create sparse matrix
    affinity_matrix = csr_matrix(
        (data_values, (row_indices, col_indices)),
        shape=(n_samples, n_samples)
    )
    
    # Make symmetric
    affinity_matrix = affinity_matrix + affinity_matrix.T
    affinity_matrix = affinity_matrix.multiply(0.5)
    
    return affinity_matrix

def build_affinity_matrix(X, batch_size=2000):
    print(f"Building affinity matrix for {X.shape[0]} samples with {X.shape[1]} features...")
    affinity_matrix = compute_sparse_affinity_batch(X, batch_size)
    
    print(f"\nAffinity matrix statistics:")
    print(f"  Shape: {affinity_matrix.shape}")
    print(f"  Non-zero elements: {affinity_matrix.nnz:,}")
    print(f"  Sparsity: {1 - affinity_matrix.nnz / (affinity_matrix.shape[0]**2):.4%}")
    print(f"  Memory usage: {affinity_matrix.data.nbytes / (1024**2):.2f} MB")
    
    return affinity_matrix

In [8]:
affinity_matrix = build_affinity_matrix(X, batch_size=2000)

Building affinity matrix for 40000 samples with 190 features...
Processing batch 1/20
Processing batch 2/20
Processing batch 3/20
Processing batch 4/20
Processing batch 5/20
Processing batch 6/20
Processing batch 7/20
Processing batch 8/20
Processing batch 9/20
Processing batch 10/20
Processing batch 11/20
Processing batch 12/20
Processing batch 13/20
Processing batch 14/20
Processing batch 15/20
Processing batch 16/20
Processing batch 17/20
Processing batch 18/20
Processing batch 19/20
Processing batch 20/20

Affinity matrix statistics:
  Shape: (40000, 40000)
  Non-zero elements: 5,564,398
  Sparsity: 99.6522%
  Memory usage: 42.45 MB


In [9]:
n_clusters = 20
clustering = SpectralClustering(
        n_clusters=n_clusters,
        affinity='precomputed',
        n_init=10,
        assign_labels='kmeans',
        eigen_solver='arpack',  # Efficient for sparse matrices
        random_state=42
    )
    
print(f"Performing spectral clustering with {n_clusters} clusters...")
labels = clustering.fit_predict(affinity_matrix)

# Print cluster distribution
unique, counts = np.unique(labels, return_counts=True)
print("\nCluster distribution:")
for i, count in enumerate(counts):
    print(f"  Cluster {i}: {count:,} samples ({count/len(labels):.1%})")

Performing spectral clustering with 20 clusters...

Cluster distribution:
  Cluster 0: 1,199 samples (3.0%)
  Cluster 1: 2,572 samples (6.4%)
  Cluster 2: 2,863 samples (7.2%)
  Cluster 3: 3,028 samples (7.6%)
  Cluster 4: 2,256 samples (5.6%)
  Cluster 5: 1,984 samples (5.0%)
  Cluster 6: 3,063 samples (7.7%)
  Cluster 7: 357 samples (0.9%)
  Cluster 8: 1,227 samples (3.1%)
  Cluster 9: 579 samples (1.4%)
  Cluster 10: 358 samples (0.9%)
  Cluster 11: 1,180 samples (2.9%)
  Cluster 12: 1,635 samples (4.1%)
  Cluster 13: 4,325 samples (10.8%)
  Cluster 14: 534 samples (1.3%)
  Cluster 15: 1,179 samples (2.9%)
  Cluster 16: 1,394 samples (3.5%)
  Cluster 17: 1,461 samples (3.7%)
  Cluster 18: 4,273 samples (10.7%)
  Cluster 19: 4,533 samples (11.3%)


In [14]:
out_df = nnary_sans78_df[:40000]
out_df['label'] = labels

/tmp/ipykernel_26419/3599618188.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out_df['label'] = labels


In [ ]:
# out_df.to_csv("examples/enhanced_spectral/cluster_results.csv")
# out_df[:1000].to_csv("examples/enhanced_spectral/cluster_results_1000.csv")

In [22]:
nnary_sans78_df.columns

Index(['uni_1_aspect', 'uni_2_aspect', 'uni_3_aspect', 'uni_4_aspect',
       'uni_5_aspect', 'uni_6_aspect', 'uni_1_count', 'uni_1_total_area',
       'uni_1_avg_x', 'uni_1_avg_y',
       ...
       'tri_3_5_6_count', 'tri_3_5_6_area_ratio_ij', 'tri_3_5_6_area_ratio_jk',
       'tri_3_5_6_avg_x', 'tri_3_5_6_avg_y', 'tri_4_5_6_count',
       'tri_4_5_6_area_ratio_ij', 'tri_4_5_6_area_ratio_jk', 'tri_4_5_6_avg_x',
       'tri_4_5_6_avg_y'],
      dtype='object', length=190)

In [23]:
TOP_N = 10

y = labels
# if X is a DataFrame, get the column names
feature_names = nnary_sans78_df.columns

top_features_per_cluster = {}

for cluster in np.unique(y):
    # make a binary target: 1 if in this cluster, 0 otherwise
    y_bin = (y == cluster).astype(int)
    
    # train a RandomForestClassifier
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X, y_bin)
    
    # get importances
    importances = pd.Series(rf.feature_importances_, index=feature_names)
    
    # sort and take top N
    top_feats = importances.sort_values(ascending=False).head(TOP_N)
    top_features_per_cluster[cluster] = top_feats

# print out the results
for cluster, feats in top_features_per_cluster.items():
    print(f"\n=== Top {TOP_N} features for cluster {cluster} ===")
    print(feats)


=== Top 10 features for cluster 0 ===
uni_3_total_area           0.280427
bi_1_3_area_ratio          0.066052
uni_1_total_area           0.049453
bi_2_3_area_ratio          0.032191
uni_2_total_area           0.029864
tri_1_3_5_area_ratio_ij    0.022142
uni_6_total_area           0.021441
uni_5_total_area           0.019069
bi_1_3_count               0.016969
uni_1_count                0.016009
dtype: float64

=== Top 10 features for cluster 1 ===
uni_3_total_area           0.117560
uni_4_total_area           0.063366
uni_1_total_area           0.052094
bi_1_3_area_ratio          0.032914
bi_2_3_area_ratio          0.028133
bi_3_4_area_ratio          0.023401
uni_2_total_area           0.019417
tri_1_3_5_area_ratio_ij    0.018410
uni_4_count                0.018062
uni_4_avg_y                0.017644
dtype: float64

=== Top 10 features for cluster 2 ===
uni_3_total_area           0.121301
bi_1_3_area_ratio          0.061962
uni_1_total_area           0.058860
uni_2_total_area         